In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 0 — CONFIG
# ─────────────────────────────────────────────────────────────

BRONZE = "saleslt.bronze"
SILVER = "saleslt.silver"

TABLES = [
    "customer",
    "product",
    "sales_order_header",
    "sales_order_detail",
    "marketingleads",
]

print("✓ Config ready")


# ─────────────────────────────────────────────────────────────
# CELL 1 — LOAD FROM BRONZE WITH inferSchema
# ─────────────────────────────────────────────────────────────

BRONZE_PATHS = {
    "customer":           "abfss://bronze@storageproject12026.dfs.core.windows.net/Sales_DB/Customer.parquet",
    "product":            "abfss://bronze@storageproject12026.dfs.core.windows.net/Sales_DB/Product.parquet",
    "sales_order_header": "abfss://bronze@storageproject12026.dfs.core.windows.net/Sales_DB/SalesOrderHeader.parquet",
    "sales_order_detail": "abfss://bronze@storageproject12026.dfs.core.windows.net/Sales_DB/SalesOrderDetail.parquet",
    "marketingleads":     "abfss://bronze@storageproject12026.dfs.core.windows.net/SalesLT/MarketingLeads/",
}

views = {}

for t, path in BRONZE_PATHS.items():
    df = spark.read \
              .format("parquet") \
              .option("inferSchema", "true") \
              .load(path)

    df.createOrReplaceTempView(f"src_{t}")
    views[t] = f"src_{t}"

    n = df.count()
    print(f"\n  ✓ {t} — {n:,} rows")
    for field in df.schema.fields:
        print(f"    {field.name:30s} → {str(field.dataType)}")

print("\n✓ CELL 1 — All tables loaded with inferred schema")


# ─────────────────────────────────────────────────────────────
# CELL 2 — RULE 1 : Trim + empty → NULL
# ─────────────────────────────────────────────────────────────

from pyspark.sql.types import StringType

for t, src in views.items():
    cols  = spark.table(src).columns
    parts = [
        f"NULLIF(TRIM(`{c}`), '') AS `{c}`"
        if isinstance(spark.table(src).schema[c].dataType, StringType)
        else f"`{c}`"
        for c in cols
    ]
    new_view = f"c1_{t}"
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {new_view} AS
        SELECT {', '.join(parts)} FROM {src}
    """)
    views[t] = new_view

print("✓ RULE 1 — Trim + empty → NULL applied")


# ─────────────────────────────────────────────────────────────
# CELL 3 — RULE 2 : Standardize casing
# ─────────────────────────────────────────────────────────────

CASING = {
    "customer": {
        "EmailAddress": "lower",
        "FirstName":    "title",
        "LastName":     "title",
    },
    "product": {
        "Color": "title",
    },
    "marketingleads": {
        "lead_status":    "title",
        "interest_level": "title",
        "country":        "upper",
        "first_name":     "title",
        "last_name":      "title",
    },
}

SQL_FN = {"lower": "LOWER", "upper": "UPPER", "title": "INITCAP"}

for t, src in views.items():
    cols       = spark.table(src).columns
    casing_map = {k.lower(): v for k, v in CASING.get(t, {}).items()}
    parts      = []
    for c in cols:
        mode = casing_map.get(c.lower())
        parts.append(
            f"{SQL_FN[mode]}(`{c}`) AS `{c}`" if mode else f"`{c}`"
        )
    new_view = f"c2_{t}"
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {new_view} AS
        SELECT {', '.join(parts)} FROM {src}
    """)
    views[t] = new_view

print("✓ RULE 2 — Casing standardized")


# ─────────────────────────────────────────────────────────────
# CELL 4 — VALIDATION 1 : NULL checks
# ─────────────────────────────────────────────────────────────

REQUIRED = {
    "customer":           ["CustomerID", "FirstName", "LastName", "EmailAddress"],
    "product":            ["ProductID",  "Name",      "ListPrice"],
    "sales_order_header": ["SalesOrderID", "CustomerID", "SubTotal", "TotalDue"],
    "sales_order_detail": ["SalesOrderID", "SalesOrderDetailID",
                           "ProductID",    "OrderQty", "LineTotal"],
    "marketingleads":     ["lead_id", "first_name", "last_name",
                           "lead_score", "lead_status"],
}

print("── VALIDATION 1 : NULL checks ───────────────────────────")
for t, src in views.items():
    for col in REQUIRED.get(t, []):
        if col not in spark.table(src).columns:
            continue
        n = spark.sql(f"""
            SELECT COUNT(*) AS n FROM {src}
            WHERE `{col}` IS NULL OR CAST(`{col}` AS STRING) = ''
        """).collect()[0]["n"]
        print(f"  {'⚠ ' if n else '✓ '} [{t}] {col}: {n} issues")


# ─────────────────────────────────────────────────────────────
# CELL 5 — VALIDATION 2 : Duplicate PKs
# ─────────────────────────────────────────────────────────────

PKS = {
    "customer":           ["CustomerID"],
    "product":            ["ProductID"],
    "sales_order_header": ["SalesOrderID"],
    "sales_order_detail": ["SalesOrderID", "SalesOrderDetailID"],
    "marketingleads":     ["lead_id"],
}

print("── VALIDATION 2 : Duplicate PKs ─────────────────────────")
for t, src in views.items():
    pk_cols = PKS.get(t, [])
    if not pk_cols:
        continue
    partition = ", ".join(f"`{c}`" for c in pk_cols)
    n = spark.sql(f"""
        SELECT COUNT(*) AS n FROM (
            SELECT COUNT(*) OVER (PARTITION BY {partition}) AS cnt
            FROM {src}
        ) WHERE cnt > 1
    """).collect()[0]["n"]
    print(f"  {'⚠ ' if n else '✓ '} [{t}] duplicates: {n}")


# ─────────────────────────────────────────────────────────────
# CELL 6 — VALIDATION 3 : Numeric checks
# ─────────────────────────────────────────────────────────────

NUMERICS = {
    "product":            [("ListPrice",    "non_negative"),
                           ("StandardCost", "non_negative")],
    "sales_order_header": [("SubTotal",     "non_negative"),
                           ("TaxAmt",       "non_negative"),
                           ("TotalDue",     "non_negative")],
    "sales_order_detail": [("OrderQty",     "positive"),
                           ("UnitPrice",    "non_negative"),
                           ("LineTotal",    "non_negative")],
    "marketingleads":     [("lead_score",      "non_negative"),
                           ("estimated_value", "non_negative")],
}

print("── VALIDATION 3 : Numeric checks ────────────────────────")
for t, src in views.items():
    checks = NUMERICS.get(t, [])
    if not checks:
        print(f"  –  [{t}] no numeric checks")
        continue
    for col, rule in checks:
        if col not in spark.table(src).columns:
            continue
        cond = f"CAST(`{col}` AS DOUBLE) < 0" \
               if rule == "non_negative" \
               else f"CAST(`{col}` AS DOUBLE) <= 0"
        n = spark.sql(f"""
            SELECT COUNT(*) AS n FROM {src}
            WHERE {cond} AND `{col}` IS NOT NULL
        """).collect()[0]["n"]
        print(f"  {'⚠ ' if n else '✓ '} [{t}] {col}: {n} violations")


# ─────────────────────────────────────────────────────────────
# CELL 7 — WRITE TO SILVER (Delta + mergeSchema)
# ─────────────────────────────────────────────────────────────

for t, src in views.items():
    df           = spark.table(src)
    table_exists = spark.catalog.tableExists(f"{SILVER}.{t}")

    if not table_exists:
        df.write \
          .format("delta") \
          .mode("overwrite") \
          .option("overwriteSchema", "true") \
          .saveAsTable(f"{SILVER}.{t}")
        print(f"  ✓ [{t}] created  — {df.count():,} rows")
    else:
        df.write \
          .format("delta") \
          .mode("overwrite") \
          .option("mergeSchema", "true") \
          .saveAsTable(f"{SILVER}.{t}")
        print(f"  ✓ [{t}] updated  — {df.count():,} rows")

print("\n✓ CELL 7 — All tables written to Silver")


# ─────────────────────────────────────────────────────────────
# CELL 8 — SUMMARY
# ─────────────────────────────────────────────────────────────

print("=" * 55)
print("  SILVER LAYER SUMMARY")
print("=" * 55)
for t in TABLES:
    n = spark.sql(
        f"SELECT COUNT(*) AS n FROM {SILVER}.{t}"
    ).collect()[0]["n"]
    print(f"  {t:35s} {n:>6,} rows")
print("=" * 55)